# vocab.collision
> Collision-strategy resolver for CogitareLink.

It chooses a *Plan* describing **how two vocabularies can coexist in
one JSON-LD document**.  Strategies are fully data-driven; the big
mapping is loaded from `data/collision_data.json` (identical to the old
`COLLISION_STRATEGIES` dict).

Public API
----------
`Resolver.choose(a:str, b:str) -> Plan`


In [ ]:
#| default_exp vocab.collision


In [ ]:
#| export
from __future__ import annotations
import json, importlib.resources as pkg
from enum import Enum
from typing import Dict, Any
from pydantic import BaseModel
from cogitarelink.core.debug import get_logger
from cogitarelink.vocab.registry import registry

log = get_logger("collision")

# bundled JSON (copy of the old COLLISION_STRATEGIES)
_TABLE: Dict[str, Any] = json.loads(
    pkg.files("cogitarelink").joinpath("data/collision_data.json").read_text()
)

In [ ]:
#| export
log = get_logger("collision")

# bundled JSON (copy of the old COLLISION_STRATEGIES)
_TABLE: Dict[str, Any] = json.loads(
    pkg.files("cogitarelink").joinpath("data/collision_data.json").read_text()
)

In [ ]:
#| export
class Strategy(str, Enum):
    property_scoped   = "property_scoped"
    graph_partition   = "graph_partition"
    property_mapping  = "property_mapping"
    nested_contexts   = "nested_contexts"
    context_versioning= "context_versioning"
    separate_graphs   = "separate_graphs"

class Plan(BaseModel):
    strategy: Strategy
    details : Dict[str, Any] = {}

In [ ]:
#| export
class Resolver:
    """Given two vocabulary prefixes, return a normalized Plan for how they should coexist.
    
    The resolver uses a lookup table of known vocabulary collision strategies.
    Each strategy defines how two vocabularies should be combined in a JSON-LD document.
    
    The lookup mechanism:
    1. Checks for an exact match between the two prefixes
    2. Tries the reverse order of prefixes
    3. Checks for wildcard rules (e.g., when one vocabulary uses @protected terms)
    4. Falls back to a default strategy (separate_graphs)
    
    Example:
        resolver = Resolver()
        plan = resolver.choose("schema", "dc")
        # Returns a Plan with strategy=Strategy.graph_partition
    """
    def __init__(self, table=None):
        """Initialize with optional custom strategy table.
        
        Args:
            table: Optional dictionary mapping vocab prefix pairs to strategy definitions.
                  If None, uses the bundled collision_data.json.
        """
        self.table = table or {tuple(k.strip("()").replace("'", "").replace('"', "").split(", ")): v 
                              for k, v in _TABLE.items() if isinstance(k, str) and k.startswith("(") and k.endswith(")")}
    
    def _lookup(self, a:str, b:str): 
        """Look up a strategy by prefix pair, trying both orderings.
        
        Args:
            a: First vocabulary prefix
            b: Second vocabulary prefix
            
        Returns:
            Strategy dict if found, None otherwise
        """
        return self.table.get((a,b)) or self.table.get((b,a))
    
    def _wildcard(self, a:str, b:str):
        """Check for wildcard rules, particularly for vocabularies with @protected terms.
        
        Args:
            a: First vocabulary prefix
            b: Second vocabulary prefix
            
        Returns:
            Strategy dict if a wildcard rule applies, None otherwise
        """
        prot = self.table.get(("*", "*_protected"))
        try:
            ent_a, ent_b = registry.by_prefix(a), registry.by_prefix(b)
            if ent_a.features.get("uses_protection") or ent_b.features.get("uses_protection"): return prot
        except KeyError: pass
        return None
    
    def choose(self, a:str, b:str) -> Plan:
        """Choose a collision strategy for two vocabulary prefixes.
        
        This is the main public API method. Given two vocabulary prefixes,
        it returns a Plan object describing how they should coexist.
        
        Args:
            a: First vocabulary prefix
            b: Second vocabulary prefix
            
        Returns:
            Plan object with strategy and details
        """
        if a==b: return Plan(strategy=Strategy.separate_graphs)
        row = self._lookup(a,b) or self._wildcard(a,b)
        if not row:
            log.debug(f"No explicit rule for ({a},{b}); using separate_graphs")
            row = dict(strategy="separate_graphs")
        return Plan(strategy=Strategy(row["strategy"]), 
                   details={k:v for k,v in row.items() if k!="strategy"})

# Global singleton instance
resolver = Resolver()


In [ ]:
#| hide
from fastcore.test import test_eq, test_ne, test_is

# explicit rule present
plan = resolver.choose("vc", "epcis")
test_eq(plan.strategy, Strategy.property_scoped)
test_eq(plan.details["property"], "credentialSubject")

# wildcard rule (protected terms) – vc + schema
plan2 = resolver.choose("vc", "schema")
test_eq(plan2.strategy, Strategy.graph_partition)

# fallback
plan3 = resolver.choose("unknown1", "unknown2")
test_eq(plan3.strategy, Strategy.separate_graphs)

# symmetry
test_is(resolver.choose("epcis","vc").strategy, Strategy.property_scoped)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()